In [15]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.opt import SolverFactory

# Importation des données
df_depots = pd.read_csv("../data/depots.csv", index_col=0)
df_zones = pd.read_csv("../data/zones.csv", index_col=0)
df_cout_transports = pd.read_csv("../data/couts_transport.csv", index_col=0)
df_parametres = pd.read_csv("../data/parametres.csv", index_col=0)
df_distances = pd.read_csv("../data/distances.csv", index_col=0)


# Paramètres
depots = list(df_depots.index)
zones = list(df_zones.index)
cout_fixe = df_depots['cout_fixe'].to_dict()
capacite = df_depots['capacite'].to_dict()
demandes = df_zones['demande'].to_dict()
nb_depot_max = df_parametres.loc['nb_depots_max', 'valeur']
budget_max = df_parametres.loc['budget_max', 'valeur']

cout_transport = {}
distances = {}
for i in depots:
    for j in zones:
        cout_transport[i, j] = df_cout_transports.loc[i,j]
        distances[i, j] = df_distances.loc[i,j]

# Modèle
model = pyo.ConcreteModel()

# Variables de décision
model.x = pyo.Var(depots, zones, bounds=(0, None), within = Reals)
model.y = pyo.Var(depots, within = Binary)

x = model.x
y = model.y

# Fonction onjectif
model.obj = pyo.Objective(expr = sum(cout_transport[i, j]*x[i, j] for i in depots for j in zones) + sum(y[i]*cout_fixe[i] for i in depots), sense = pyo.minimize)

# Contraintes
model.c1 = pyo.ConstraintList()
for j in zones:
    model.c1.add(expr = sum(x[i, j] for i in depots) == demandes[j])

model.c2 = pyo.ConstraintList()
for i in depots:
    model.c2.add(expr = sum(x[i, j] for j in zones) <= capacite[i]*y[i])

model.c3 = pyo.Constraint(expr = sum(y[i] for i in depots) <= nb_depot_max)

model.c4 = pyo.Constraint(
    expr=sum(cout_fixe[i]*y[i] for i in depots) + 
         sum(cout_transport[i,j]*x[i,j] for i in depots for j in zones) 
         <= budget_max
)

# Résolution
opt = SolverFactory('glpk')
opt.solve(model)

print(f"Coût total : {pyo.value(model.obj):,.0f} FCFA\n")

depots_ouverts = [i for i in depots if pyo.value(y[i]) == 1]
print(f"Dépôts ouverts ({len(depots_ouverts)}) : {', '.join(depots_ouverts)}\n")

for i in depots_ouverts:
    charge = sum(pyo.value(x[i, j]) for j in zones if pyo.value(x[i, j]) > 0)
    print(f"Dépôt {i} — Charge : {charge:.0f}/{capacite[i]} ")
    for j in zones:
        if pyo.value(x[i, j]) > 0:
            print(f"{j} : {pyo.value(x[i, j]):.0f} ")
    print()

Coût total : 5,620,420 FCFA

Dépôts ouverts (3) : Parakou, Abomey, Natitingou

Dépôt Parakou — Charge : 335/400 
Kandi : 80 
Bembéréké : 80 
Nikki : 90 
Savalou : 75 
Bohicon : 10 

Dépôt Abomey — Charge : 300/300 
Bohicon : 85 
Lokossa : 85 
Ouidah : 70 
Comè : 60 

Dépôt Natitingou — Charge : 250/250 
Malanville : 120 
Kandi : 20 
Djougou : 110 



In [16]:
import folium

carte = folium.Map(location=[9.0, 2.3], zoom_start=7)

couleurs = {
    'Parakou': 'blue',
    'Abomey': 'green',
    'Natitingou': 'red',
    'Cotonou': 'purple'
}

for i in depots:
    if pyo.value(y[i]) == 1:
        folium.Marker(
            location=[df_depots.loc[i, 'latitude'], df_depots.loc[i, 'longitude']],
            popup=f"<b>Dépôt : {i}</b><br>Capacité : {capacite[i]} <br>Coût fixe : {cout_fixe[i]:,} FCFA",
            icon=folium.Icon(color=couleurs.get(i, 'gray'), icon='home')
        ).add_to(carte)

for j in zones:
    folium.CircleMarker(
        location=[df_zones.loc[j, 'latitude'], df_zones.loc[j, 'longitude']],
        radius=8,
        color='orange',
        fill=True,
        fill_color='orange',
        fill_opacity=0.7,
        popup=f"<b>Zone : {j}</b><br>Demande : {demandes[j]} "
    ).add_to(carte)


for i in depots:
    if pyo.value(y[i]) == 1:
        for j in zones:
            val = pyo.value(x[i, j])
            if val and val > 0:
                folium.PolyLine(
                    locations=[
                        [df_depots.loc[i, 'latitude'], df_depots.loc[i, 'longitude']],
                        [df_zones.loc[j, 'latitude'], df_zones.loc[j, 'longitude']]
                    ],
                    color=couleurs.get(i, 'gray'),
                    weight=max(1, val/20),  # épaisseur proportionnelle au flux
                    opacity=0.7,
                    popup=f"{i} → {j} : {val:.0f} t"
                ).add_to(carte)


legende = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 10px; border-radius: 8px;
     border: 2px solid grey; font-size: 13px;">
    <b>Légende</b><br>
    🏠 Dépôt ouvert<br>
    🟠 Zone sinistrée<br>
    — Flux de livraison
</div>
"""
carte.get_root().html.add_child(folium.Element(legende))


carte.save('../outputs/carte_humanitaire.html')
# Afficher dans Jupyter
carte